In [1]:
learner_profiles_df = spark.table(
    "demo.silver.learner_profiles"
)

print(
    "Silver learner profile rows:",
    learner_profiles_df.count()
)

learner_profiles_df.show(
    truncate=False
)

Silver learner profile rows: 4


+--------+-----------------+------------------+----------------+---------------------------------------------------------+----------------+-------------------+-------------------+----------+
|user_id |registration_date|preferred_language|background_level|learning_goal                                            |main_domain     |profile_updated_at |ingestion_time     |is_current|
+--------+-----------------+------------------+----------------+---------------------------------------------------------+----------------+-------------------+-------------------+----------+
|user_001|2026-07-01       |English           |Intermediate    |Strengthen virtual memory and operating systems knowledge|Computer Science|2026-07-23 18:10:00|2026-07-23 18:15:00|true      |
|user_001|2026-07-01       |English           |Beginner        |Improve understanding of operating systems               |Computer Science|2026-07-01 08:00:00|2026-07-01 08:05:00|false     |
|user_002|2026-07-03       |English          

In [2]:
from pyspark.sql.functions import col
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank

user_key_window = Window.orderBy(
    col("user_id")
)

dim_learner_df = (
    learner_profiles_df
    .filter(
        col("is_current") == True
    )
    .withColumn(
        "user_key",
        dense_rank().over(user_key_window)
    )
    .select(
        col("user_key").cast("int"),
        col("user_id"),
        col("registration_date"),
        col("preferred_language"),
        col("background_level"),
        col("learning_goal"),
        col("main_domain"),
        col("profile_updated_at"),
        col("is_current").alias("is_active")
    )
)

dim_learner_df.orderBy(
    "user_key"
).show(truncate=False)

26/07/28 15:03:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/28 15:03:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/28 15:03:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/28 15:03:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/28 15:03:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+--------+--------+-----------------+------------------+----------------+---------------------------------------------------------+----------------+-------------------+---------+
|user_key|user_id |registration_date|preferred_language|background_level|learning_goal                                            |main_domain     |profile_updated_at |is_active|
+--------+--------+-----------------+------------------+----------------+---------------------------------------------------------+----------------+-------------------+---------+
|1       |user_001|2026-07-01       |English           |Intermediate    |Strengthen virtual memory and operating systems knowledge|Computer Science|2026-07-23 18:10:00|true     |
|2       |user_002|2026-07-03       |English           |Intermediate    |Improve programming fundamentals and recursion           |Computer Science|2026-07-03 09:30:00|true     |
|3       |user_003|2026-07-05       |English           |Intermediate    |Learn technical concepts through

In [3]:
spark.sql("""
DELETE FROM demo.gold.dim_learner
""")

DataFrame[]

In [4]:
dim_learner_df.writeTo(
    "demo.gold.dim_learner"
).append()

26/07/28 15:05:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/28 15:05:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/28 15:05:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/28 15:05:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/28 15:05:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [5]:
spark.sql("""
SELECT *
FROM demo.gold.dim_learner
ORDER BY user_key
""").show(truncate=False)

+--------+--------+-----------------+------------------+----------------+---------------------------------------------------------+----------------+-------------------+---------+
|user_key|user_id |registration_date|preferred_language|background_level|learning_goal                                            |main_domain     |profile_updated_at |is_active|
+--------+--------+-----------------+------------------+----------------+---------------------------------------------------------+----------------+-------------------+---------+
|1       |user_001|2026-07-01       |English           |Intermediate    |Strengthen virtual memory and operating systems knowledge|Computer Science|2026-07-23 18:10:00|true     |
|2       |user_002|2026-07-03       |English           |Intermediate    |Improve programming fundamentals and recursion           |Computer Science|2026-07-03 09:30:00|true     |
|3       |user_003|2026-07-05       |English           |Intermediate    |Learn technical concepts through

In [6]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT user_key) AS distinct_user_keys,
    COUNT(DISTINCT user_id) AS distinct_user_ids
FROM demo.gold.dim_learner
""").show()

+----------+------------------+-----------------+
|total_rows|distinct_user_keys|distinct_user_ids|
+----------+------------------+-----------------+
|         3|                 3|                3|
+----------+------------------+-----------------+



In [7]:
taxonomy_df = spark.table(
    "demo.silver.content_taxonomy"
)

print(
    "Silver taxonomy rows:",
    taxonomy_df.count()
)

taxonomy_df.select(
    "taxonomy_id",
    "domain",
    "topic",
    "subtopic",
    "concept_name",
    "normalized_domain",
    "normalized_topic",
    "normalized_subtopic",
    "normalized_concept_name",
    "taxonomy_level",
    "parent_taxonomy_id",
    "first_detected_at",
    "validation_status",
    "is_active"
).orderBy(
    "taxonomy_level",
    "domain",
    "topic",
    "subtopic",
    "concept_name"
).show(truncate=False)

Silver taxonomy rows: 10
+----------------------------------------------------------------+----------------+-----------------+---------------------+--------------+-----------------+-----------------+---------------------+-----------------------+--------------+----------------------------------------------------------------+-------------------+-----------------+---------+
|taxonomy_id                                                     |domain          |topic            |subtopic             |concept_name  |normalized_domain|normalized_topic |normalized_subtopic  |normalized_concept_name|taxonomy_level|parent_taxonomy_id                                              |first_detected_at  |validation_status|is_active|
+----------------------------------------------------------------+----------------+-----------------+---------------------+--------------+-----------------+-----------------+---------------------+-----------------------+--------------+------------------------------------------

In [8]:
from pyspark.sql.functions import (
    col,
    when,
    concat_ws,
    regexp_replace,
    dense_rank
)
from pyspark.sql.window import Window

active_taxonomy_df = taxonomy_df.filter(
    (col("is_active") == True)
    & col("validation_status").isin(
        "approved",
        "pending"
    )
)

topic_key_window = Window.orderBy(
    col("taxonomy_id")
)

dim_topic_base_df = (
    active_taxonomy_df
    .withColumn(
        "topic_key",
        dense_rank().over(topic_key_window).cast("int")
    )
    .withColumn(
        "topic_name",
        when(
            col("taxonomy_level") == "domain",
            col("domain")
        )
        .when(
            col("taxonomy_level") == "topic",
            col("topic")
        )
        .when(
            col("taxonomy_level") == "subtopic",
            col("subtopic")
        )
        .otherwise(
            col("concept_name")
        )
    )
    .withColumn(
        "normalized_topic_name",
        when(
            col("taxonomy_level") == "domain",
            col("normalized_domain")
        )
        .when(
            col("taxonomy_level") == "topic",
            col("normalized_topic")
        )
        .when(
            col("taxonomy_level") == "subtopic",
            col("normalized_subtopic")
        )
        .otherwise(
            col("normalized_concept_name")
        )
    )
    .withColumn(
        "topic_id",
        concat_ws(
            "_",
            col("taxonomy_level"),
            regexp_replace(
                col("normalized_topic_name"),
                " ",
                "_"
            )
        )
    )
)

In [9]:
parent_keys_df = dim_topic_base_df.select(
    col("taxonomy_id").alias("parent_taxonomy_id_lookup"),
    col("topic_key").alias("parent_topic_key")
)

dim_topic_df = (
    dim_topic_base_df.alias("child")
    .join(
        parent_keys_df.alias("parent"),
        col("child.parent_taxonomy_id")
        == col("parent.parent_taxonomy_id_lookup"),
        "left"
    )
    .select(
        col("child.topic_key"),
        col("child.taxonomy_id"),
        col("child.topic_id"),
        col("child.topic_name"),
        col("child.normalized_topic_name"),
        col("child.domain"),
        col("parent.parent_topic_key").cast("int"),
        col("child.taxonomy_level"),
        col("child.first_detected_at"),
        col("child.validation_status"),
        col("child.is_active")
    )
)

dim_topic_df.orderBy(
    "topic_key"
).show(truncate=False)

26/07/28 15:10:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/28 15:10:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/28 15:10:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/28 15:10:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/28 15:10:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/28 15:10:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/28 1

+---------+----------------------------------------------------------------+------------------------------+---------------------+---------------------+----------------+----------------+--------------+-------------------+-----------------+---------+
|topic_key|taxonomy_id                                                     |topic_id                      |topic_name           |normalized_topic_name|domain          |parent_topic_key|taxonomy_level|first_detected_at  |validation_status|is_active|
+---------+----------------------------------------------------------------+------------------------------+---------------------+---------------------+----------------+----------------+--------------+-------------------+-----------------+---------+
|1        |3af7fa6cc86c65a77274c1065a16dbe8d864aad92950628ce54693753bc84a7a|domain_computer_science       |Computer Science     |computer science     |Computer Science|NULL            |domain        |2026-07-19 12:00:00|approved         |true     |
|2  

26/07/28 15:10:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/28 15:10:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [10]:
spark.sql("""
DELETE FROM demo.gold.dim_topic
""")

DataFrame[]

In [11]:
dim_topic_df.writeTo(
    "demo.gold.dim_topic"
).append()

26/07/28 15:11:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/28 15:11:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/28 15:11:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/28 15:11:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/28 15:11:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/28 15:11:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/28 1

In [12]:
spark.sql("""
SELECT
    topic_key,
    topic_id,
    topic_name,
    taxonomy_level,
    parent_topic_key,
    validation_status,
    is_active
FROM demo.gold.dim_topic
ORDER BY topic_key
""").show(truncate=False)

+---------+------------------------------+---------------------+--------------+----------------+-----------------+---------+
|topic_key|topic_id                      |topic_name           |taxonomy_level|parent_topic_key|validation_status|is_active|
+---------+------------------------------+---------------------+--------------+----------------+-----------------+---------+
|1        |domain_computer_science       |Computer Science     |domain        |NULL            |approved         |true     |
|2        |concept_base_case             |Base Case            |concept       |9               |approved         |true     |
|3        |concept_process_memory        |Process Memory       |concept       |8               |approved         |true     |
|4        |concept_memory_layout         |Memory Layout        |concept       |8               |pending          |true     |
|5        |subtopic_virtual_memory       |Virtual Memory       |subtopic      |10              |approved         |true     |


In [13]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT topic_key) AS distinct_topic_keys,
    COUNT(DISTINCT taxonomy_id) AS distinct_taxonomy_ids,
    COUNT(DISTINCT topic_id) AS distinct_topic_ids
FROM demo.gold.dim_topic
""").show()

+----------+-------------------+---------------------+------------------+
|total_rows|distinct_topic_keys|distinct_taxonomy_ids|distinct_topic_ids|
+----------+-------------------+---------------------+------------------+
|        10|                 10|                   10|                10|
+----------+-------------------+---------------------+------------------+



In [14]:
from pyspark.sql import Row

content_types_data = [
    Row(
        content_type_key=1,
        content_type_id="verbal",
        content_type_name="Verbal",
        category="verbal",
        academic_orientation="humanistic",
        requires_logic=False,
        requires_numerical_reasoning=False,
        requires_text_interpretation=True,
        requires_visual_reasoning=False,
        requires_memorization=False,
        description="Content that primarily requires reading and textual interpretation."
    ),
    Row(
        content_type_key=2,
        content_type_id="visual",
        content_type_name="Visual",
        category="visual",
        academic_orientation="mixed",
        requires_logic=False,
        requires_numerical_reasoning=False,
        requires_text_interpretation=False,
        requires_visual_reasoning=True,
        requires_memorization=False,
        description="Content that relies on diagrams, graphs, layouts, or visual reasoning."
    ),
    Row(
        content_type_key=3,
        content_type_id="quantitative",
        content_type_name="Quantitative",
        category="quantitative",
        academic_orientation="realistic",
        requires_logic=True,
        requires_numerical_reasoning=True,
        requires_text_interpretation=False,
        requires_visual_reasoning=False,
        requires_memorization=False,
        description="Content that requires numerical reasoning and calculation."
    ),
    Row(
        content_type_key=4,
        content_type_id="logical",
        content_type_name="Logical",
        category="practical",
        academic_orientation="realistic",
        requires_logic=True,
        requires_numerical_reasoning=False,
        requires_text_interpretation=False,
        requires_visual_reasoning=False,
        requires_memorization=False,
        description="Content that primarily requires logical reasoning and structured problem solving."
    ),
    Row(
        content_type_key=5,
        content_type_id="memory_based",
        content_type_name="Memory Based",
        category="verbal",
        academic_orientation="mixed",
        requires_logic=False,
        requires_numerical_reasoning=False,
        requires_text_interpretation=True,
        requires_visual_reasoning=False,
        requires_memorization=True,
        description="Content where recall or memorization is central."
    ),
    Row(
        content_type_key=6,
        content_type_id="mixed",
        content_type_name="Mixed",
        category="mixed",
        academic_orientation="mixed",
        requires_logic=True,
        requires_numerical_reasoning=False,
        requires_text_interpretation=True,
        requires_visual_reasoning=True,
        requires_memorization=False,
        description="Content combining multiple cognitive and content characteristics."
    )
]

dim_content_type_df = spark.createDataFrame(content_types_data)

dim_content_type_df.orderBy(
    "content_type_key"
).show(truncate=False)

[Stage 40:=================>                                      (5 + 11) / 16]

+----------------+---------------+-----------------+------------+--------------------+--------------+----------------------------+----------------------------+-------------------------+---------------------+---------------------------------------------------------------------------------+
|content_type_key|content_type_id|content_type_name|category    |academic_orientation|requires_logic|requires_numerical_reasoning|requires_text_interpretation|requires_visual_reasoning|requires_memorization|description                                                                      |
+----------------+---------------+-----------------+------------+--------------------+--------------+----------------------------+----------------------------+-------------------------+---------------------+---------------------------------------------------------------------------------+
|1               |verbal         |Verbal           |verbal      |humanistic          |false         |false                       |

In [15]:
from pyspark.sql.functions import col

dim_content_type_ready_df = dim_content_type_df.select(
    col("content_type_key").cast("int"),
    col("content_type_id").cast("string"),
    col("content_type_name").cast("string"),
    col("category").cast("string"),
    col("academic_orientation").cast("string"),
    col("requires_logic").cast("boolean"),
    col("requires_numerical_reasoning").cast("boolean"),
    col("requires_text_interpretation").cast("boolean"),
    col("requires_visual_reasoning").cast("boolean"),
    col("requires_memorization").cast("boolean"),
    col("description").cast("string")
)

In [16]:
spark.sql("""
DELETE FROM demo.gold.dim_content_type
""")

DataFrame[]

In [17]:
dim_content_type_ready_df.writeTo(
    "demo.gold.dim_content_type"
).append()

In [18]:
spark.sql("""
SELECT *
FROM demo.gold.dim_content_type
ORDER BY content_type_key
""").show(truncate=False)

+----------------+---------------+-----------------+------------+--------------------+--------------+----------------------------+----------------------------+-------------------------+---------------------+---------------------------------------------------------------------------------+
|content_type_key|content_type_id|content_type_name|category    |academic_orientation|requires_logic|requires_numerical_reasoning|requires_text_interpretation|requires_visual_reasoning|requires_memorization|description                                                                      |
+----------------+---------------+-----------------+------------+--------------------+--------------+----------------------------+----------------------------+-------------------------+---------------------+---------------------------------------------------------------------------------+
|1               |verbal         |Verbal           |verbal      |humanistic          |false         |false                       |

In [19]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_type_key) AS distinct_content_type_keys,
    COUNT(DISTINCT content_type_id) AS distinct_content_type_ids
FROM demo.gold.dim_content_type
""").show()

+----------+--------------------------+-------------------------+
|total_rows|distinct_content_type_keys|distinct_content_type_ids|
+----------+--------------------------+-------------------------+
|         6|                         6|                        6|
+----------+--------------------------+-------------------------+



In [20]:
reference_materials_df = spark.table(
    "demo.silver.reference_materials"
)

print(
    "Silver reference material rows:",
    reference_materials_df.count()
)

reference_materials_df.show(
    truncate=False
)

Silver reference material rows: 6
+-------------+--------------------+----------------------+-----------------------------------+-------------------------------+-------------------+-------------------+----------------+---------------------------------+-----------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------+------------------------------+----------------------------------------------------------------+---------+
|reference_id |batch_id            |source_type           |source_name                        |file_name                      |import_time        |ingestion_time     |domain          |title                            |topic            |content_text                                                                                                                  

In [21]:
from pyspark.sql.functions import col, dense_rank
from pyspark.sql.window import Window

reference_key_window = Window.orderBy(
    col("reference_id")
)

dim_reference_source_df = (
    reference_materials_df
    .filter(
        col("is_active") == True
    )
    .withColumn(
        "reference_key",
        dense_rank().over(reference_key_window).cast("int")
    )
    .select(
        col("reference_key"),
        col("reference_id"),
        col("source_name"),
        col("source_type"),
        col("file_name"),
        col("reliability_level"),
        col("domain"),
        col("is_active")
    )
)

dim_reference_source_df.orderBy(
    "reference_key"
).show(truncate=False)

+-------------+-------------+-----------------------------------+----------------------+-------------------------------+-----------------+----------------+---------+
|reference_key|reference_id |source_name                        |source_type           |file_name                      |reliability_level|domain          |is_active|
+-------------+-------------+-----------------------------------+----------------------+-------------------------------+-----------------+----------------+---------+
|1            |reference_001|Operating Systems Course           |lecture_slides        |virtual_memory_lecture.pdf     |official         |Computer Science|true     |
|2            |reference_002|Operating Systems Course           |lecture_slides        |virtual_memory_lecture.pdf     |official         |Computer Science|true     |
|3            |reference_003|Operating Systems Course Repository|official_documentation|virtual_memory_notes.md        |official         |Computer Science|true     |
|4  

26/07/28 15:16:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/28 15:16:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/28 15:16:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/28 15:16:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/28 15:16:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
